# 02 - Deep Learning: Embedding + Bi-LSTM (TensorFlow / Keras)
**BrandPulse AI - Phase 2 (Approach B) & Phase 3 (comparison)**

Unlike TF-IDF, which ignores word order, an LSTM reads the tweet token by token and can learn that "not good" differs from "good".

Pipeline: **clean text -> tokenize -> pad -> Embedding -> Bi-LSTM -> Dense (softmax)**.
At the end we compare against the classical model from notebook 01 on the *same* test set.

> A GPU is recommended (Colab/Kaggle work fine) but this dataset (~11k tweets) also trains on a CPU in a few minutes.

## 1. Setup

In [1]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

sys.path.append(str(Path.cwd().parent))
from src.data import LABELS, SEED, load_airline, make_split
from src.preprocess import clean_tweet, ensure_nltk
from src.tokenizer import SimpleTokenizer, pad_sequences
from src.evaluate import evaluate, plot_confusion_matrix

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ensure_nltk()
keras.utils.set_random_seed(SEED)

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "Tweets.csv"
MODELS_DIR, REPORTS_DIR = ROOT / "models", ROOT / "reports"
for d in (MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

C:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


TensorFlow 2.21.0 | GPUs: []


## 2. Load, split and clean
Same split as notebook 01. For the LSTM we use a **lighter clean**: we keep stop words because function words ("not", "but", "very") carry the context the LSTM is supposed to learn.

In [ ]:
df = load_airline(DATA_PATH)
train_df, test_df = make_split(df)
train_df, test_df = train_df.copy(), test_df.copy()

light = lambda t: clean_tweet(t, remove_stopwords=False)
train_df["clean_text"] = train_df["text"].apply(light)
test_df["clean_text"] = test_df["text"].apply(light)

label2id = {l: i for i, l in enumerate(LABELS)}
y_train = train_df["label"].map(label2id).values
y_test = test_df["label"].map(label2id).values
print(train_df.shape, test_df.shape, label2id)
train_df[["text", "clean_text"]].head()

## 3. Tokenize and pad
Each word is mapped to an integer id (most frequent `MAX_WORDS` words; the rest become `<OOV>`), then every sequence is padded/truncated to the same length.
`MAX_LEN` is the 95th percentile of tweet length so we don't waste computation on a few very long tweets.

In [ ]:
MAX_WORDS, EMBED_DIM, BATCH_SIZE, EPOCHS = 20_000, 128, 64, 15

tokenizer = SimpleTokenizer(num_words=MAX_WORDS).fit_on_texts(train_df["clean_text"])   # fit on TRAIN only
lengths = train_df["clean_text"].str.split().str.len()
MAX_LEN = int(np.percentile(lengths, 95))
print("Vocabulary size:", tokenizer.vocab_size, "| MAX_LEN:", MAX_LEN)

plt.figure(figsize=(7, 3.5))
plt.hist(lengths, bins=40, color="#4e79a7"); plt.axvline(MAX_LEN, color="red", ls="--", label="MAX_LEN (95th pct)")
plt.xlabel("tokens per tweet"); plt.legend(); plt.show()

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df["clean_text"]), MAX_LEN)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df["clean_text"]), MAX_LEN)

print("\nExample:")
print("text  :", train_df["clean_text"].iloc[0])
print("padded:", X_train[0])

## 4. Model architecture
`Embedding` (learns a dense vector per word) -> `SpatialDropout1D` -> **Bidirectional LSTM** (reads the tweet forwards and backwards)
-> `Dense` -> `Softmax` over 3 classes. Padding positions are masked so they don't influence the LSTM.

In [ ]:
def build_model(vocab_size, max_len, n_classes=3):
    model = keras.Sequential([
        layers.Input(shape=(max_len,)),
        layers.Embedding(vocab_size, EMBED_DIM, mask_zero=True),
        layers.SpatialDropout1D(0.3),
        layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model(tokenizer.vocab_size, MAX_LEN)
model.summary()

## 5. Train
* **Class weights** compensate for the imbalanced classes.
* **EarlyStopping** monitors *validation loss* and restores the best weights, which prevents overfitting.
* **ReduceLROnPlateau** halves the learning rate when validation loss stalls.

In [ ]:
cw = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=y_train)
class_weight = dict(enumerate(cw))
print("Class weights:", {LABELS[k]: round(v, 2) for k, v in class_weight.items()})

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.1, epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight, callbacks=callbacks, verbose=2,
)

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], marker="o", title="Loss")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], marker="o", title="Accuracy")
for ax in axes: ax.set_xlabel("epoch")
plt.tight_layout()
fig.savefig(REPORTS_DIR / "lstm_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

If `val_loss` rises while `loss` keeps falling, the model is overfitting - early stopping rolls back to the best epoch.

## 6. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report

test_probs = model.predict(X_test, batch_size=256, verbose=0)
y_pred = np.array(LABELS)[test_probs.argmax(axis=1)]
y_true = test_df["label"].values

lstm_metrics = evaluate(y_true, y_pred, "BiLSTM")
print(classification_report(y_true, y_pred, digits=3))
fig = plot_confusion_matrix(y_true, y_pred, LABELS, "Embedding + Bi-LSTM",
                            save_path=REPORTS_DIR / "cm_lstm.png")
plt.show()

## 7. Classical vs Deep Learning - performance report

In [ ]:
classical_path = REPORTS_DIR / "classical_results.json"
if classical_path.exists():
    classical = pd.DataFrame(json.loads(classical_path.read_text()))
else:
    print("classical_results.json not found - run notebook 01 first.")
    classical = pd.DataFrame()

comparison = pd.concat([classical, pd.DataFrame([lstm_metrics])], ignore_index=True)
comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
(REPORTS_DIR / "lstm_results.json").write_text(json.dumps(lstm_metrics, indent=2))
display(comparison.set_index("Model").style.highlight_max(color="#c6efce"))

In [ ]:
long = comparison.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(11, 4.5))
sns.barplot(data=long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1); plt.title("Model comparison (macro-averaged metrics, same test set)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Side-by-side confusion matrices (needs cm_classical.png from notebook 01)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
for ax, f, title in zip(axes, ["cm_classical.png", "cm_lstm.png"], ["Classical (TF-IDF + LogReg)", "Deep Learning (BiLSTM)"]):
    p = REPORTS_DIR / f
    if p.exists():
        ax.imshow(plt.imread(p))
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Error analysis - where does the LSTM fail?

In [ ]:
errs = test_df.assign(pred=y_pred, confidence=test_probs.max(axis=1).round(3))
errs = errs[errs["label"] != errs["pred"]].sort_values("confidence", ascending=False)
print(f"{len(errs)} errors. Most confident mistakes:")
errs[["text", "label", "pred", "confidence"]].head(10)

## 9. Custom inference

In [ ]:
def predict_lstm(texts):
    cleaned = [clean_tweet(t, remove_stopwords=False) for t in texts]
    X = pad_sequences(tokenizer.texts_to_sequences(cleaned), MAX_LEN)
    p = model.predict(X, verbose=0)
    out = pd.DataFrame(p, columns=LABELS).round(3)
    out.insert(0, "prediction", np.array(LABELS)[p.argmax(1)])
    out.insert(0, "text", texts)
    return out

predict_lstm([
    "The service was amazing!",
    "I waited 4 hours just to get a cold burger.",
    "Does anyone know the baggage allowance for international flights?",
    "Oh great, another delayed flight. Just what I needed.",      # sarcasm
    "Not happy with the delay but the crew was very helpful.",    # mixed
])

## 10. Save the model, tokenizer and config for the Streamlit app

In [ ]:
model.save(MODELS_DIR / "lstm_model.keras")
tokenizer.to_json(MODELS_DIR / "tokenizer.json")
(MODELS_DIR / "lstm_config.json").write_text(json.dumps(
    {"max_len": MAX_LEN, "max_words": MAX_WORDS, "labels": LABELS}, indent=2))
print(sorted(p.name for p in MODELS_DIR.glob("*")))

## 11. Discussion: interpretability vs performance
Fill in after you have your numbers:

| | TF-IDF + Logistic Regression | Embedding + Bi-LSTM |
|---|---|---|
| Training time | seconds | minutes (GPU helps) |
| Inference speed | very fast | slower |
| Interpretability | high - coefficients show which words drive each class | low - black box |
| Word order / context | ignored (bigrams help a little) | captured |
| Data hunger | works with small data | benefits from more data |
| Sarcasm | struggles | struggles too, but handles negation better |

**Next steps:** pretrained embeddings (GloVe), a fine-tuned transformer (DistilBERT/RoBERTa), or a larger dataset such as Sentiment140.